# Patent Hit Probability — citation percentile within Sector × grant-year cohorts

Computes each US utility patent's **citation percentile** within its **WIPO technology sector ×
grant year** cohort, for each window (C3 / C5 / C10 / C_all). A percentile of 0.99 = top 1%, 0.95 = top 5%,
0.90 = top 10%. Primary key: `patent_id`.

## Raw / input data (directory & structure)
```
/project/jevans/Dawoon/Science of Science/PatentView/output/patent_citation.parquet   # patent_id, C_3, C_5, C_10, C_all (forward citations per window)
/project/jevans/Dawoon/Science of Science/PatentView/output/patent_metadata.parquet   # patent_id, grant_year
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_wipo_technology.tsv.zip   # patent_id, wipo_field_sequence, wipo_sector_title
```
Cohort **Sector** = `wipo_sector_title` (5 sectors: Electrical engineering, Instruments, Chemistry,
Mechanical engineering, Other fields), taken as the patent's primary WIPO field (lowest sequence).

## Metric (definition)
Within each (Sector, grant_year) cohort, `pctl_{w}` is the citation **percentile** in `[0, 1]`
(`rank(pct=True, ascending=True, method='min')`): higher = more cited, zero-citation ties sit near 0 and
the most-cited near 1. Read hits as: top 1% ⇔ `pctl ≥ 0.99`, top 5% ⇔ `≥ 0.95`, top 10% ⇔ `≥ 0.90`.
Computed independently for `C_3, C_5, C_10, C_all`.

## Output
`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_hit_probability.parquet` — `patent_id, wipo_sector, grant_year` +
`pctl_{w}` for w ∈ {c3,c5,c10,call} (citation percentile in [0,1] within the cohort).

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
POUT = pv.OUT
OUT_FP = pv.out('patent_hit_probability.parquet')
pv.preflight('patent_hit_probability')

ROOT: C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science


In [2]:
# Every citation count in patent_citation.parquet gets its own percentile, not just the four
# granted windows. Which columns those are is read from the FILE rather than listed here, so
# a column added upstream (uniqueC_*, say) is picked up without editing this notebook.
import pyarrow.parquet as pq

CIT_FP = os.path.join(POUT, 'patent_citation.parquet')
ALL_COLS = pq.ParquetFile(CIT_FP).schema_arrow.names
COUNT_COLS = [c for c in ALL_COLS
              if c.startswith(('C_', 'appC', 'uniqueC')) or c in ('C_3', 'C_5', 'C_10', 'C_all')]
COUNT_COLS = [c for c in COUNT_COLS if c != 'patent_id']
print(f'{len(COUNT_COLS)} citation columns to rank:')
for fam in ('C_', 'appC', 'uniqueC'):
    sub = [c for c in COUNT_COLS if c.startswith(fam)]
    print(f'  {fam:<10}{len(sub):>3}  {sub[:6]}{" …" if len(sub) > 6 else ""}')

# The four the rest of the project already reads by their short names.
LEGACY = {'pctl_c3': 'C_3', 'pctl_c5': 'C_5', 'pctl_c10': 'C_10', 'pctl_call': 'C_all'}


def add_percentile(df, cohort_cols, cols):
    """Within each cohort partition, the citation PERCENTILE per column in [0, 1]
    (higher = more cited). rank(ascending=True, method='min') puts zero-citation ties near 0
    and the top near 1; e.g. top 1% / 5% / 10% are pctl >= 0.99 / 0.95 / 0.90.

    Ranked WITHIN (sector, grant year) because a raw count compares a 1980 patent with a 2020
    one and mostly measures how long it has had to accumulate citations."""
    g = df.groupby(cohort_cols)
    for c in cols:
        df[f'pctl_{c}'] = g[c].rank(pct=True, ascending=True, method='min').astype('float32')
    for short, col in LEGACY.items():
        if col in cols:
            df[short] = df[f'pctl_{col}']
    return df

## 1. Load citations + grant year + WIPO sector

In [3]:
%%time
cit = pd.read_parquet(CIT_FP, columns=['patent_id'] + COUNT_COLS)
yr = pd.read_parquet(os.path.join(POUT, 'patent_metadata.parquet'),
                     columns=['patent_id', 'grant_year'])
wp = pd.read_csv(os.path.join(D, 'g_wipo_technology.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'wipo_field_sequence', 'wipo_sector_title'],
                 dtype={'patent_id': str, 'wipo_sector_title': str})
wp['wipo_field_sequence'] = pd.to_numeric(wp['wipo_field_sequence'], errors='coerce')
sector = (wp.sort_values(['patent_id', 'wipo_field_sequence'])
            .drop_duplicates('patent_id', keep='first')
            .set_index('patent_id')['wipo_sector_title'].rename('wipo_sector'))
df = (yr.merge(cit, on='patent_id', how='left')
        .merge(sector, left_on='patent_id', right_index=True, how='left'))
for c in COUNT_COLS:
    # a patent absent from patent_citation.parquet was cited zero times, not unknown
    df[c] = df[c].fillna(0).astype('int64')
df = df.dropna(subset=['wipo_sector'])
print(f'patents with a WIPO sector: {len(df):,} | cohorts '
      f'{df.groupby(["wipo_sector", "grant_year"]).ngroups:,}')
del cit, yr, wp, sector; gc.collect()

patents with sector: 8,512,715
CPU times: total: 27.5 s
Wall time: 28.2 s


## 2. Compute percentiles + save

In [4]:
%%time
df = add_percentile(df, ['wipo_sector', 'grant_year'], COUNT_COLS)
PCTL = [f'pctl_{c}' for c in COUNT_COLS]
out = df[['patent_id', 'wipo_sector', 'grant_year'] + PCTL + list(LEGACY)]
out.to_parquet(OUT_FP, index=False, compression='zstd')
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.0f} MB)')

print(f'\npercentile summary, one row per citation kind')
print(f'  {"column":<28}{"mean":>9}{"median":>9}{"= 0 ties":>10}{"top 1%":>9}')
print('  ' + '-' * 65)
for c in COUNT_COLS:
    v = out[f'pctl_{c}']
    print(f'  {c:<28}{v.mean():>9.4f}{v.median():>9.4f}'
          f'{(v <= v.min()).mean()*100:>9.1f}%{(v >= .99).mean()*100:>8.2f}%')
print('\n  A high "= 0 ties" share is not a fault: most patents receive no citations of that')
print('  kind at all, so they share the lowest rank. It does mean a percentile cut inside')
print('  that block is decided by the tie-breaking rule, not by the data.')
print(f'\n  legacy aliases kept: {list(LEGACY)}')
display(out.head(6))

WROTE C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science\notebook\patent\output\patent_hit_probability.parquet  (8,512,715 rows, 7 cols)
citation percentile summary:


            pctl_c3       pctl_c5      pctl_c10     pctl_call
count  8.512715e+06  8.512715e+06  8.512715e+06  8.512715e+06
mean   3.075000e-01  3.572000e-01  3.878000e-01  3.979000e-01
std    3.631000e-01  3.550000e-01  3.467000e-01  3.446000e-01
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
50%    1.000000e-04  3.159000e-01  3.726000e-01  3.843000e-01
75%    6.487000e-01  6.814000e-01  7.062000e-01  7.131000e-01
max    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00


,patent_id,wipo_sector,grant_year,pctl_c3,pctl_c5,pctl_c10,pctl_call
0,10000000,Instruments,2018,0.899643,0.922765,0.934123,0.934123
1,10000001,Mechanical engineering,2018,0.000020,0.000020,0.000020,0.000020
2,10000002,Chemistry,2018,0.000024,0.000024,0.000024,0.000024
3,10000003,Mechanical engineering,2018,0.607858,0.687702,0.599118,0.599118
4,10000004,Mechanical engineering,2018,0.000020,0.000020,0.000020,0.000020
5,10000005,Mechanical engineering,2018,0.000020,0.000020,0.000020,0.000020
6,10000006,Mechanical engineering,2018,0.607858,0.473649,0.394487,0.394487
7,10000007,Mechanical engineering,2018,0.938078,0.959226,0.937536,0.937536


CPU times: total: 14.8 s
Wall time: 15.3 s
